In [ ]:
!pip install deep-sort-realtime
!pip install ultralytics
!pip install datetime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 105.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
 

Importing Libraries

In [ ]:
import datetime  # Library for handling date and time operations
from ultralytics import YOLO  # Library for loading and using the YOLO model
import cv2  # OpenCV library for image and video processing
from deep_sort_realtime.deepsort_tracker import DeepSort  # Library for the DeepSORT tracker
from google.colab.patches import cv2_imshow  # Library for displaying images in Google Colab

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
def create_video_writer(video_cap,output_filename):
  frame_width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
  frame_height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
  fps = int(video_cap.get(cv2.CAP_PROP_FPS))
  fourcc = cv2.VideoWriter_fourcc(*"MP4V")
  writer = cv2.VideoWriter(output_filename,fourcc,fps,(frame_width,frame_height))
  return writer

In [ ]:
CONFIDENCE_THRESHOLD = 0.8  # Confidence threshold for detecting objects
GREEN = (0, 255, 0)  # Color for drawing bounding boxes
WHITE = (255, 255, 255)  # Color for drawing text

video_cap = cv2.VideoCapture("/content/biking.mp4")  # Initialize the video capture object to read the video
writer = create_video_writer(video_cap, "output.mp4")  # Initialize the video writer object to save the processed video

model = YOLO("yolov8n.pt")  # Load the pre-trained YOLOv8n model
tracker = DeepSort(max_age=50)  # Initialize the DeepSORT tracker

Process Video Frames

Start Frame Processing Loop

In [ ]:
while True:
  start = datetime.datetime.now()
  ret,frame = video_cap.read()  # Read a frame from the video
  if not ret:
    break

Run YOLO Model for Object Detection

In [ ]:
detections = model(frame)[0]
results = []

for data in detections.boxes.data.tolist():
  confidence = data[4]
  if float(confidence) < CONFIDENCE_THRESHOLD:
    continue
  xmin, ymin, xmax, ymax = int(data[0]), int(data[1]), int(data[2]), int(data[3])
  class_id = int(data[5])
  results.append([[xmin, ymin, xmax - xmin, ymax - ymin], confidence, class_id])

WARNING ⚠️ 'source' is missing. Using 'source=/usr/local/lib/python3.11/dist-packages/ultralytics/assets'.

image 1/2 /usr/local/lib/python3.11/dist-packages/ultralytics/assets/bus.jpg: 640x480 4 persons, 1 bus, 1 stop sign, 8.2ms
image 2/2 /usr/local/lib/python3.11/dist-packages/ultralytics/assets/zidane.jpg: 384x640 2 persons, 1 tie, 7.1ms
Speed: 2.7ms preprocess, 7.6ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)


In [ ]:
print(type(frame))
print(frame.shape if frame is not None else "Frame is None")


<class 'NoneType'>
Frame is None


In [ ]:
import cv2
import datetime
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

CONFIDENCE_THRESHOLD = 0.8
GREEN = (0, 255, 0)
WHITE = (255, 255, 255)

# Load YOLOv8 model
model = YOLO("yolov8n.pt")

# Initialize DeepSORT tracker
tracker = DeepSort(max_age=50)

# Open the video
video_cap = cv2.VideoCapture("/content/biking.mp4")

# Function to create a video writer
def create_video_writer(cap, output_filename):
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    return cv2.VideoWriter(output_filename, fourcc, fps, (width, height))

# Create video writer
writer = create_video_writer(video_cap, "output.mp4")

# Main loop
while True:
    start = datetime.datetime.now()
    ret, frame = video_cap.read()
    if not ret:
        break

    # Run YOLOv8 on the current frame
    detections = model(frame)[0]
    results = []

    # Process YOLO detections
    for data in detections.boxes.data.tolist():
        confidence = data[4]
        if float(confidence) < CONFIDENCE_THRESHOLD:
            continue
        xmin, ymin, xmax, ymax = int(data[0]), int(data[1]), int(data[2]), int(data[3])
        class_id = int(data[5])
        results.append([[xmin, ymin, xmax - xmin, ymax - ymin], confidence, class_id])

    # Update the DeepSORT tracker
    tracks = tracker.update_tracks(results, frame=frame)

    # Draw bounding boxes and track IDs
    for track in tracks:
        if not track.is_confirmed():
            continue
        track_id = track.track_id
        xmin, ymin, xmax, ymax = map(int, track.to_ltrb())
        cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), GREEN, 2)
        cv2.rectangle(frame, (xmin, ymin - 20), (xmin + 40, ymin), GREEN, -1)
        cv2.putText(frame, f"{track_id}", (xmin + 5, ymin - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, WHITE, 2)

    # Write the frame to the output video
    writer.write(frame)

# Release everything
video_cap.release()
writer.release()
cv2.destroyAllWindows()



0: 640x384 1 person, 1 motorcycle, 73.0ms
Speed: 2.5ms preprocess, 73.0ms inference, 5.3ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 person, 1 motorcycle, 11.6ms
Speed: 2.4ms preprocess, 11.6ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 person, 1 motorcycle, 12.6ms
Speed: 2.9ms preprocess, 12.6ms inference, 2.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 person, 1 motorcycle, 31.7ms
Speed: 2.6ms preprocess, 31.7ms inference, 2.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 person, 1 motorcycle, 33.2ms
Speed: 2.8ms preprocess, 33.2ms inference, 10.4ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 person, 1 motorcycle, 17.7ms
Speed: 3.1ms preprocess, 17.7ms inference, 2.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 person, 1 motorcycle, 7.1ms
Speed: 2.5ms preprocess, 7.1ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 person,

error: OpenCV(4.11.0) /io/opencv/modules/highgui/src/window.cpp:1295: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvDestroyAllWindows'
